# Lab 2 — Imbalance Analysis

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

> **Checkpoints:** imbalance **99:1** · baseline accuracy **0.99** · baseline F1 (fraud) = **0.00**

<!-- cisco-day06-lab02-expanded-2026 -->

## The accuracy trap

With **1%** fraud, always predicting legit yields **99%** accuracy but **0** fraud detections.

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

fraud_count = int(y.sum())
legit_count = len(y) - fraud_count
imbalance_ratio = legit_count / max(fraud_count, 1)

print("Lab 2 — Imbalance analysis")
print(f"total rows: {len(y)}")
print(f"fraud: {fraud_count}, legit: {legit_count}")
print(f"imbalance ratio (legit:fraud): {imbalance_ratio:.1f}:1")
print(f"fraud rate: {y.mean():.4f}")


## Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = y.value_counts().sort_index()
sns.barplot(x=counts.index.map({0: "legit", 1: "fraud"}), y=counts.values, ax=ax, palette="Set2")
ax.set_ylabel("count")
ax.set_title("Class imbalance (full dataset)")
plt.tight_layout()
plt.show()


## Stratified train/test split

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"test:  {len(X_test)} (fraud {int(y_test.sum())})")

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "total": [len(y_train), len(y_test)],
    "fraud": [int(y_train.sum()), int(y_test.sum())],
    "legit": [int((y_train == 0).sum()), int((y_test == 0).sum())],
})
display(split_summary)


## Majority-class baseline

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

baseline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", DummyClassifier(strategy="most_frequent")),
    ]
)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

baseline_acc = (y_pred == y_test).mean()
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"baseline accuracy (majority class): {baseline_acc:.4f}")
print(f"baseline F1 (fraud): {f1:.4f}")
print(f"unique predictions: {sorted(set(y_pred.tolist()))}")


## Classification report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

## Metrics cheat sheet

In [ ]:
metrics_help = pd.DataFrame({
    "metric": ["accuracy", "precision (fraud)", "recall (fraud)", "F1 (fraud)"],
    "baseline": [f"{baseline_acc:.4f}", "0.00", "0.00", f"{f1:.4f}"],
    "higher_better": ["yes", "yes", "yes", "yes"],
    "use_when": [
        "balanced classes only",
        "false alarms costly",
        "missing fraud costly",
        "balance precision & recall",
    ],
})
display(metrics_help)


### Imbalance prompt 1

Show fraud rate in train and test.

In [ ]:
print({'train': round(y_train.mean(),4), 'test': round(y_test.mean(),4)})

### Imbalance prompt 2

Count predictions per class.

In [ ]:
print(pd.Series(y_pred).value_counts().to_dict())

### Imbalance prompt 3

Compute confusion-style counts manually.

In [ ]:
tp = int(((y_pred==1)&(y_test==1)).sum()); print({'tp': tp, 'fn': int(y_test.sum())-tp})

### Imbalance prompt 4

Display class weights intuition.

In [ ]:
print({'legit': legit_count, 'fraud': fraud_count, 'ratio': round(imbalance_ratio,1)})

### Imbalance prompt 5

Plot fraud rate bar.

In [ ]:
rates = pd.Series({'train': y_train.mean(), 'test': y_test.mean()}); ax = rates.plot(kind='bar', figsize=(4,3)); ax.set_ylabel('fraud rate');

### Imbalance prompt 6

List unique merchant categories in train.

In [ ]:
print(sorted(X_train['merchant_category'].unique())[:8])

### Imbalance prompt 7

Show mean amount by class in train.

In [ ]:
tmp = X_train.copy(); tmp['is_fraud']=y_train.values; print(tmp.groupby('is_fraud')['amount'].mean().round(2).to_dict())

### Imbalance prompt 8

Show mean distance by class in train.

In [ ]:
print(tmp.groupby('is_fraud')['distance_from_home'].mean().round(2).to_dict())

### Imbalance prompt 9

Check baseline predicted probabilities shape.

In [ ]:
print(baseline.predict_proba(X_test).shape)

### Imbalance prompt 10

Verify baseline always predicts 0.

In [ ]:
print(set(y_pred.tolist()))

### Imbalance prompt 11

Compute legit accuracy contribution.

In [ ]:
print(round((y_test==0).mean(), 4))

### Imbalance prompt 12

Summarize why F1 is zero.

In [ ]:
print('F1=0 because precision and recall for fraud are both 0.')

### Imbalance prompt 13

Compare accuracy vs fraud recall.

In [ ]:
print({'accuracy': round(baseline_acc,4), 'fraud_recall': 0.0})

### Imbalance prompt 14

Display train fraud indices count.

In [ ]:
print(int(y_train.sum()))

### Imbalance prompt 15

Display test fraud indices count.

In [ ]:
print(int(y_test.sum()))

### Imbalance prompt 16

Recompute imbalance ratio.

In [ ]:
print(round((y==0).sum()/max(y.sum(),1), 1))

### Imbalance prompt 17

Show first 5 test labels.

In [ ]:
print(y_test.head().tolist())

### Imbalance prompt 18

Show first 5 baseline predictions.

In [ ]:
print(y_pred[:5].tolist())

### Imbalance prompt 19

State business risk framing.

In [ ]:
print('Missed fraud often costs more than false alarms in payments.')

### Imbalance prompt 20

Bridge to resampling.

In [ ]:
print('Lab 3 oversamples minority fraud on train only.')

### Lab 2 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 2 recap step 1: completed")

### Lab 2 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 2 recap step 2: completed")

### Lab 2 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 2 recap step 3: completed")

### Lab 2 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 2 recap step 4: completed")

## Final checkpoint

In [ ]:
assert fraud_count == 10
assert abs(imbalance_ratio - 99.0) < 0.1
assert abs(baseline_acc - 0.99) < 0.01
assert f1 == 0.0
assert int(y_test.sum()) == 2
print("✓ All checkpoint assertions passed")


## Reflection

Why is accuracy misleading when fraud is 1% of rows?